# Assignment 2: DeepFake Image Detection
### Course: Foundations of Data Science (FDS)
**Topic**: Forensic Analysis of Residual Noise Texture in Digital Images for Deepfake Detection  
**Reference Paper**: *Forensics Analysis of Residual Noise Texture in Digital Images for Detection of Deepfake*, Arthur Méreur, Antoine Mallet, Rémi Cogranne, Minoru Kuribayashi (IEEE ICASSP 2025)  
**Methodology**: Discrete Laplacian Residuals, Multi-scale Fractal Dimension (FD), Block-wise Texture Complexity (TC), and Novel Frequency-Domain Residual Ensembles.

---
## 1. Environment Setup & Configuration
This notebook runs seamlessly in either **Local Execution** or **Google Colab**.


In [ ]:
# Check environment and configure paths
import os
import sys

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    print("[*] Running in Google Colab environment.")
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q opencv-python-headless scikit-learn matplotlib seaborn tqdm
    PROJECT_DIR = '/content/drive/MyDrive/Deepfake_FDS/Actual_Project'
else:
    print("[*] Running in local environment.")
    PROJECT_DIR = os.getcwd()

print(f"Project working directory: {PROJECT_DIR}")


## 2. Core Forensic Feature Extraction Modules
We implement:
1. **Laplacian Noise Residual**: Discrete 2D convolution with 3x3 kernel:
$$\begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$$
2. **Fractal Dimension (FD)**: Multi-scale box-counting over 44 scales between $2 < D < 112$.
3. **Texture Complexity (TC)**: Block-wise ($D=16$) complexity:
$$t(R) = 1 - \frac{1}{D^2} \sum_{m,n} \frac{2^{-r_{m,n}} r_{m,n}}{1}, \quad TC = \log\left(\frac{t(R)}{1 - t(R)}\right) + 4$$
4. **Novelty - Radial Spectral Profile**: 2D FFT azimuthal radial power spectrum capturing AI synthesis grid artifacts.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

LAPLACIAN_KERNEL = np.array([
    [0,  1,  0],
    [1, -4,  1],
    [0,  1,  0]
], dtype=np.float32)

def compute_laplacian_residual(image):
    img_float = image.astype(np.float32)
    return cv2.filter2D(img_float, -1, LAPLACIAN_KERNEL)

def compute_box_count(residual, threshold=0.01, num_scales=44):
    Z = np.abs(residual)
    h, w = Z.shape
    D_values = np.unique(np.round(np.logspace(np.log10(3), np.log10(110), num=num_scales)).astype(int))
    counts = []
    for D in D_values:
        grid_h, grid_w = max(1, h // D), max(1, w // D)
        sub_sampled = cv2.resize(Z, (grid_w, grid_h), interpolation=cv2.INTER_AREA)
        counts.append(max(1, np.sum(sub_sampled > threshold)))
    log_counts = np.log(np.array(counts, dtype=np.float64))
    coeffs = np.polyfit(-np.log(D_values), log_counts, 1)
    return float(coeffs[0]), log_counts

def compute_texture_complexity(residual, D=16):
    R = np.abs(residual).astype(np.float64)
    h, w = R.shape
    bh, bw = h // D, w // D
    tc_map = np.zeros((bh, bw), dtype=np.float64)
    D2 = float(D * D)
    for bi in range(bh):
        for bj in range(bw):
            block = R[bi * D : (bi + 1) * D, bj * D : (bj + 1) * D]
            val = (2.0 ** (-block)) * block
            t = np.clip(1.0 - (np.sum(val) / D2), 1e-6, 1.0 - 1e-6)
            tc_map[bi, bj] = np.log(t / (1.0 - t)) + 4.0
    tc_vector = tc_map.flatten()
    tc_stats = np.array([np.mean(tc_vector), np.std(tc_vector), np.median(tc_vector)])
    return tc_map, tc_vector, tc_stats

def compute_spectral_residual_features(residual, num_radial_bins=32):
    R = residual.astype(np.float64)
    h, w = R.shape
    F = np.fft.fftshift(np.fft.fft2(R))
    log_spec = np.log1p(np.abs(F) ** 2)
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    r = np.hypot(x - cx, y - cy)
    edges = np.linspace(0, min(cy, cx), num_radial_bins + 1)
    profile = [float(np.mean(log_spec[(r >= edges[i]) & (r < edges[i + 1])])) if np.any((r >= edges[i]) & (r < edges[i + 1])) else 0.0 for i in range(num_radial_bins)]
    return np.array(profile)

print("[OK] Forensic extraction routines compiled successfully.")


## 3. Exploratory Data Analysis & Residual Visualizations
Visualizing original images alongside their high-frequency Laplacian noise residuals.


In [ ]:
from src.dataset import get_image_paths

real_paths = get_image_paths(os.path.join(PROJECT_DIR, 'data', 'preprocessed', 'real'))
fake_paths = get_image_paths(os.path.join(PROJECT_DIR, 'data', 'preprocessed', 'fake'))

print(f"Found {len(real_paths)} Real preprocessed images and {len(fake_paths)} Fake preprocessed images.")

if real_paths and fake_paths:
    r_img = cv2.imread(real_paths[0], cv2.IMREAD_GRAYSCALE)
    f_img = cv2.imread(fake_paths[0], cv2.IMREAD_GRAYSCALE)
    r_res = compute_laplacian_residual(r_img)
    f_res = compute_laplacian_residual(f_img)
    
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    axes[0, 0].imshow(r_img, cmap='gray'); axes[0, 0].set_title('Real Photograph (Luminance)'); axes[0, 0].axis('off')
    im1 = axes[0, 1].imshow(r_res, cmap='seismic', vmin=-30, vmax=30); axes[0, 1].set_title('Real: Laplacian Noise Residual'); axes[0, 1].axis('off')
    fig.colorbar(im1, ax=axes[0, 1], fraction=0.046)
    
    axes[1, 0].imshow(f_img, cmap='gray'); axes[1, 0].set_title('AI Deepfake (Luminance)'); axes[1, 0].axis('off')
    im2 = axes[1, 1].imshow(f_res, cmap='seismic', vmin=-30, vmax=30); axes[1, 1].set_title('AI Deepfake: Laplacian Residual'); axes[1, 1].axis('off')
    fig.colorbar(im2, ax=axes[1, 1], fraction=0.046)
    plt.tight_layout()
    plt.show()


## 4. Multi-Scale Feature Distribution Analysis
Comparing Fractal Dimension (FD) and Texture Complexity (TC) distributions between Real and Deepfake images.


In [ ]:
feat_dir = os.path.join(PROJECT_DIR, 'data', 'features')
y = np.load(os.path.join(feat_dir, 'y.npy'))
X_compact = np.load(os.path.join(feat_dir, 'X_paper_compact.npy'))

fd_real = X_compact[y == 0, 0]
fd_fake = X_compact[y == 1, 0]
tc_real = X_compact[y == 0, 1]
tc_fake = X_compact[y == 1, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(fd_real, bins=25, alpha=0.65, label='Real Photos', color='#2b5c8f', edgecolor='black')
axes[0].hist(fd_fake, bins=25, alpha=0.65, label='AI Deepfakes', color='#d95f02', edgecolor='black')
axes[0].set_title('Fractal Dimension (FD) Distribution')
axes[0].set_xlabel('Estimated Fractal Dimension')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].hist(tc_real, bins=25, alpha=0.65, label='Real Photos', color='#2b5c8f', edgecolor='black')
axes[1].hist(tc_fake, bins=25, alpha=0.65, label='AI Deepfakes', color='#d95f02', edgecolor='black')
axes[1].set_title('Texture Complexity (TC) Distribution')
axes[1].set_xlabel('Mean Texture Complexity')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Machine Learning Benchmarking & 5-Fold Cross-Validation
We cross-validate multiple models:
1. **Paper Linear SVM**: Replicating IEEE ICASSP 2025 Table I.
2. **ML Baselines**: Random Forest, RBF Kernel SVM, Logistic Regression.
3. **Novel Hybrid Ensemble**: Combining multi-scale fractal geometry, texture moments, and high-frequency spectral profiles via soft voting.


In [ ]:
from src.models import get_paper_svm, get_baseline_models, get_novel_ensemble
from src.evaluate import evaluate_model_cv, format_benchmark_table

X_stats = np.load(os.path.join(feat_dir, 'X_paper_stats.npy'))
X_novel = np.load(os.path.join(feat_dir, 'X_novel_feature.npy'))

benchmark_models = {
    "Paper Linear SVM (FD + TC)": (get_paper_svm(C=1.0), X_compact, "FD Slope + TC Mean (Paper)"),
    "Baseline Random Forest": (get_baseline_models()["Random Forest"], X_stats, "FD + TC Statistical Descriptors"),
    "Baseline RBF SVM": (get_baseline_models()["RBF SVM"], X_stats, "FD + TC Statistical Descriptors"),
    "Novel Hybrid Ensemble": (get_novel_ensemble(), X_novel, "FD + TC + Spectral + Moments (Novel)")
}

results = {}
for name, (model, X_data, feat_label) in benchmark_models.items():
    print(f"Evaluating {name} with 5-fold CV...")
    res = evaluate_model_cv(model, X_data, y, n_splits=5)
    res["feature_name"] = feat_label
    results[name] = res

from IPython.display import Markdown
Markdown(format_benchmark_table(results))


## 6. Performance Evaluation: ROC Curves & Confusion Matrix


In [ ]:
from src.visualize import plot_roc_curves, plot_confusion_matrix

# Plot ROC Curves
plt.figure(figsize=(8, 7))
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Chance (AUC = 0.50)')
for name, res in results.items():
    roc = res['roc']
    auc = res['summary']['roc_auc']
    plt.plot(roc['fpr'], roc['tpr'], lw=2.5, label=f"{name} (AUC = {auc:.3f})")
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('5-Fold Cross-Validated ROC Performance')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Plot Confusion Matrix for Novel Ensemble
cm = results["Novel Hybrid Ensemble"]["cm"]
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap=plt.cm.Blues)
plt.title("Confusion Matrix: Novel Hybrid Ensemble")
plt.colorbar()
classes = ['Real', 'Fake']
plt.xticks([0, 1], classes)
plt.yticks([0, 1], classes)
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i, j]:.1f}", ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontweight="bold")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


## 7. Forensic Inference On Unseen Images
Test any arbitrary image to predict whether it is Real or AI-generated with confidence score and forensic reasoning.


In [ ]:
def predict_deepfake(image_path, model_pipeline, feature_extractor_type="novel"):
    from src.features import extract_all_features_for_image
    
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not load image from {image_path}")
    if img.shape != (512, 512):
        img = cv2.resize(img, (512, 512))
        
    res = compute_laplacian_residual(img)
    feats = extract_all_features_for_image(res)
    
    if feature_extractor_type == "novel":
        X_test = feats["novel_feature"].reshape(1, -1)
    else:
        X_test = feats["paper_compact"].reshape(1, -1)
        
    pred = model_pipeline.predict(X_test)[0]
    prob = model_pipeline.predict_proba(X_test)[0]
    
    label = "AI-Generated / DeepFake" if pred == 1 else "Authentic Real Photograph"
    confidence = prob[pred] * 100
    
    print("="*50)
    print("FORENSIC DEEPFAKE DETECTION VERDICT")
    print("="*50)
    print(f"File: {os.path.basename(image_path)}")
    print(f"Prediction : {label}")
    print(f"Confidence : {confidence:.2f}%")
    print(f"P(Real)    : {prob[0]*100:.2f}%")
    print(f"P(Fake)    : {prob[1]*100:.2f}%")
    print(f"Estimated FD: {feats['fd_slope']:.4f}")
    print(f"Mean TC     : {feats['tc_stats'][0]:.4f}")
    print("="*50)

# Demonstrate inference on first test sample
best_model = benchmark_models["Novel Hybrid Ensemble"][0]
best_model.fit(X_novel, y)
predict_deepfake(fake_paths[0], best_model, feature_extractor_type="novel")
